## Skrypt tworzący DataFrame ze zdenormalizowanym zbiorem AIdev-pop
#### Dane z wielu tabeli zebrane do jednej
#### Denormalizacja w celu szybszego przetwarzania
#### Bazą jest tabela pull_request, do niej dodane takie kolumny jak
#### liczba linii zmienionych (i oddzielnie dodanych/usuniętych), plików, commitów w PRze,
#### średni, maksymalny rozmiar commita w PRze
#### ilość ocen (reviews), akceptacji (approvals), komentarzy pod Issue (od ludzi)
#### czy PR został zmergowany, czas od utworzenia do zmergowania itp

In [ ]:
"""
Build a denormalized PR-level DataFrame from the AIDev-pop subset
(repos with >100 stars) for analyzing what practices correlate with
the quality of Agentic-PRs.

Requirements:
    pip install pandas pyarrow huggingface_hub
"""

import pandas as pd

BASE = "hf://datasets/hao-li/AIDev"

# ------------------------------------------------------------------ #
# 1. Load AIDev-pop tables                                            #
# ------------------------------------------------------------------ #
pr           = pd.read_parquet(f"{BASE}/pull_request.parquet")
repo         = pd.read_parquet(f"{BASE}/repository.parquet")
commits      = pd.read_parquet(f"{BASE}/pr_commits.parquet")
commit_files = pd.read_parquet(f"{BASE}/pr_commit_details.parquet")
reviews      = pd.read_parquet(f"{BASE}/pr_reviews.parquet")
review_cmts  = pd.read_parquet(f"{BASE}/pr_review_comments_v2.parquet")
issue_cmts   = pd.read_parquet(f"{BASE}/pr_comments.parquet")
task_type    = pd.read_parquet(f"{BASE}/pr_task_type.parquet")
rel_issue    = pd.read_parquet(f"{BASE}/related_issue.parquet")
print("All parquets downloaded")

# Parse timestamps (PR table)
for col in ["created_at", "closed_at", "merged_at"]:
    pr[col] = pd.to_datetime(pr[col], errors="coerce", utc=True)

# ------------------------------------------------------------------ #
# 2. Outcome / quality features (dependent variables)                 #
# ------------------------------------------------------------------ #
pr["is_merged"]          = pr["merged_at"].notna().astype(int)
pr["is_closed_unmerged"] = ((pr["state"] == "closed") & pr["merged_at"].isna()).astype(int)
pr["time_to_merge_hours"] = (pr["merged_at"] - pr["created_at"]).dt.total_seconds() / 3600
pr["time_to_close_hours"] = (pr["closed_at"] - pr["created_at"]).dt.total_seconds() / 3600

# ------------------------------------------------------------------ #
# 3. PR-size features (aggregate pr_commit_details to PR level)       #
#    NB: pr_commit_details has ONE ROW PER FILE PER COMMIT.           #
# ------------------------------------------------------------------ #
size_feats = (
    commit_files.groupby("pr_id")
    .agg(
        loc_added    = ("additions", "sum"),
        loc_deleted  = ("deletions", "sum"),
        loc_churn    = ("changes",   "sum"),
        files_changed= ("filename",  "nunique"),
    )
    .reset_index()
)

# ------------------------------------------------------------------ #
# 4. Commit-granularity features                                      #
# ------------------------------------------------------------------ #
# Per-commit size first (sum file-level changes within a sha)
per_commit_size = (
    commit_files.groupby(["pr_id", "sha"])
    .agg(commit_total=("changes", "sum"))
    .reset_index()
)
commit_size_stats = (
    per_commit_size.groupby("pr_id")
    .agg(
        mean_commit_size = ("commit_total", "mean"),
        std_commit_size  = ("commit_total", "std"),
        max_commit_size  = ("commit_total", "max"),
    )
    .reset_index()
)

# Commit counts + message length from pr_commits
granularity_feats = (
    commits.groupby("pr_id")
    .agg(
        n_commits           = ("sha",     "nunique"),
        mean_commit_msg_len = ("message", lambda s: s.fillna("").str.len().mean()),
    )
    .reset_index()
)

# ------------------------------------------------------------------ #
# 5. Review-effort features                                           #
# ------------------------------------------------------------------ #
n_reviews = reviews.groupby("pr_id").size().rename("n_reviews").reset_index()

n_changes_requested = (
    reviews[reviews["state"] == "CHANGES_REQUESTED"]
    .groupby("pr_id").size().rename("n_changes_requested").reset_index()
)
n_approvals = (
    reviews[reviews["state"] == "APPROVED"]
    .groupby("pr_id").size().rename("n_approvals").reset_index()
)

# Inline review comments: count via review -> pr_id (works whether or
# not pr_review_comments_v2 carries pr_id directly)
if "pr_id" in review_cmts.columns:
    n_inline_cmts = (
        review_cmts.groupby("pr_id").size()
        .rename("n_inline_review_comments").reset_index()
    )
else:
    n_inline_cmts = (
        review_cmts
        .merge(reviews[["id", "pr_id"]].rename(columns={"id": "pull_request_review_id"}),
               on="pull_request_review_id", how="left")
        .groupby("pr_id").size()
        .rename("n_inline_review_comments").reset_index()
    )

# Issue-thread comments from humans only (excludes bots, including the agent itself)
human_cmts = issue_cmts[issue_cmts["user_type"] == "User"]
n_issue_cmts = (
    human_cmts.groupby("pr_id").size()
    .rename("n_human_issue_comments").reset_index()
)

# ------------------------------------------------------------------ #
# 6. Task type and linked-issue indicator                             #
# ------------------------------------------------------------------ #
task_feats = task_type[["id", "type"]].rename(columns={"id": "pr_id", "type": "task_type"})

has_issue = (
    rel_issue.groupby("pr_id").size()
    .rename("n_linked_issues").reset_index()
)

# ------------------------------------------------------------------ #
# 7. PR-description features                                          #
# ------------------------------------------------------------------ #
pr["title_len"] = pr["title"].fillna("").str.len()
pr["body_len"]  = pr["body"].fillna("").str.len()

print("PR-description features set")

# ------------------------------------------------------------------ #
# 8. Build the master denormalized DataFrame                          #
# ------------------------------------------------------------------ #
df = pr.rename(columns={"id": "pr_id"}).copy()

# Attach repo metadata (controls / stratifiers)
df = df.merge(
    repo.rename(columns={
        "id": "repo_id",
        "language": "repo_language",
        "stars":    "repo_stars",
        "forks":    "repo_forks",
        "license":  "repo_license",
    })[["repo_id", "repo_language", "repo_stars", "repo_forks", "repo_license"]],
    on="repo_id", how="left",
)

# Left-join every PR-level aggregate
for table in [
    size_feats, granularity_feats, commit_size_stats,
    n_reviews, n_changes_requested, n_approvals,
    n_inline_cmts, n_issue_cmts,
    task_feats, has_issue,
]:
    df = df.merge(table, on="pr_id", how="left")

# Count features: NaN really means "zero of these happened"
count_cols = [
    "n_reviews", "n_changes_requested", "n_approvals",
    "n_inline_review_comments", "n_human_issue_comments",
    "n_linked_issues", "n_commits", "files_changed",
    "loc_added", "loc_deleted", "loc_churn",
]
df[count_cols] = df[count_cols].fillna(0).astype("int64")
df["has_linked_issue"] = (df["n_linked_issues"] > 0).astype(int)

# ------------------------------------------------------------------ #
# 9. Sanity check + save                                              #
# ------------------------------------------------------------------ #
assert df["pr_id"].is_unique, "Row explosion in joins — investigate!"
print(f"Final shape           : {df.shape}")
print(f"PRs in pull_request   : {len(pr)}")
print(f"Merge rate            : {df['is_merged'].mean():.2%}")
print(f"Agents                : {df['agent'].value_counts().to_dict()}")
print(df.head())

df.to_parquet("aidev_pop_pr_features.parquet", index=False)
print("\nSaved -> aidev_pop_pr_features.parquet")